# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [1]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH_V3.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-04-01 12:11:31.073 | INFO     | __main__:<module>:11 - SDM en DWH connecties succesvol.


Dim_Fiets (SCD Type 2) MEES

In [8]:
logger.info("Start ETL Dim_Fiets (SCD Type 2 met business key fietsnr + source_id)")

# =========================================================
# 1. BRONDATA OPHALEN
# =========================================================

df_inkoop = pd.read_sql_query("""
SELECT 
    f.fietsnr, f.soort, f.merk, f.type, f.kleur,
    fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
FROM Fiets_Inkoop_Fiets f
JOIN Fiets_Inkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)

df_verkoop = pd.read_sql_query("""
SELECT 
    f.fietsnr, f.soort, f.merk, f.type, f.kleur,
    fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
FROM Fiets_Verkoop_Fiets f
JOIN Fiets_Verkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)

df_onderhoud = pd.read_sql_query("""
SELECT 
    f.fietsnr, f.soort, f.merk, f.type, f.kleur,
    fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
FROM Onderhoud_Fiets f
JOIN Onderhoud_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)

# source_id toevoegen
df_inkoop["source_id"] = "Fiets_Inkoop"
df_verkoop["source_id"] = "Fiets_Verkoop"
df_onderhoud["source_id"] = "Onderhoud"

# samenvoegen
df_fiets_source = pd.concat(
    [df_inkoop, df_verkoop, df_onderhoud],
    ignore_index=True
)

logger.info(f"Aantal bronrecords: {len(df_fiets_source)}")

# =========================================================
# 2. DUBBELE CHECK (BINNEN ZELFDE BUSINESS KEY)
# =========================================================

duplicate_bk = df_fiets_source.duplicated(
    subset=["fietsnr", "source_id"], keep=False
)

if duplicate_bk.any():
    logger.warning("Dubbele records binnen dezelfde business key gevonden")

# =========================================================
# 3. ACTUELE DWH DATA
# =========================================================

df_dwh = pd.read_sql_query("""
SELECT *
FROM Dim_Fiets
WHERE eindtijd IS NULL
""", dwh_conn)

logger.info(f"Aantal actuele fietsen in DWH: {len(df_dwh)}")

# =========================================================
# 4. WIJZIGINGSCHECK
# =========================================================

def fiets_is_changed(src, dwh):
    return (
        str(src["soort"]) != str(dwh["soort"]) or
        str(src["merk"]) != str(dwh["merk"]) or
        str(src["type"]) != str(dwh["type"]) or
        str(src["kleur"]) != str(dwh["kleur"]) or
        str(src["fabrikantnr"]) != str(dwh["fabrikantnr"]) or
        str(src["naam"]) != str(dwh["fabrikantnaam"]) or
        str(src["adres"]) != str(dwh["fabrikantadres"]) or
        str(src["plaats"]) != str(dwh["fabrikantplaats"])
    )

# =========================================================
# 5. PREVIEW
# =========================================================

new_count = 0
changed_count = 0
unchanged_count = 0

for _, src in df_fiets_source.iterrows():
    match = df_dwh[
        (df_dwh["fietsnr"] == src["fietsnr"]) &
        (df_dwh["source_id"] == src["source_id"])
    ]

    if match.empty:
        new_count += 1
    else:
        if fiets_is_changed(src, match.iloc[0]):
            changed_count += 1
        else:
            unchanged_count += 1

logger.info(f"Preview | new={new_count}, changed={changed_count}, unchanged={unchanged_count}")

# =========================================================
# 6. ETL (SCD TYPE 2)
# =========================================================

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted = 0
updated = 0
unchanged = 0

for _, src in df_fiets_source.iterrows():

    fietsnr = int(src["fietsnr"])
    source_id = src["source_id"]

    match = df_dwh[
        (df_dwh["fietsnr"] == fietsnr) &
        (df_dwh["source_id"] == source_id)
    ]

    # NIEUW
    if match.empty:
        dwh_conn.execute("""
        INSERT INTO Dim_Fiets (
            fietsnr, soort, merk, type, kleur,
            fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
            begintijd, eindtijd, source_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, ?)
        """, (
            fietsnr,
            src["soort"],
            src["merk"],
            src["type"],
            src["kleur"],
            int(src["fabrikantnr"]),
            src["naam"],
            src["adres"],
            src["plaats"],
            now,
            source_id
        ))

        inserted += 1

    # BESTAAND
    else:
        dwh_row = match.iloc[0]

        if fiets_is_changed(src, dwh_row):

            # afsluiten
            dwh_conn.execute("""
            UPDATE Dim_Fiets
            SET eindtijd = ?
            WHERE fiets_sk = ?
            """, (now, int(dwh_row["fiets_sk"])))

            # nieuwe versie
            dwh_conn.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
                begintijd, eindtijd, source_id
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, ?)
            """, (
                fietsnr,
                src["soort"],
                src["merk"],
                src["type"],
                src["kleur"],
                int(src["fabrikantnr"]),
                src["naam"],
                src["adres"],
                src["plaats"],
                now,
                source_id
            ))

            updated += 1
        else:
            unchanged += 1

# =========================================================
# 7. COMMIT
# =========================================================

dwh_conn.commit()

logger.success(
    f"Dim_Fiets klaar | inserted={inserted}, updated={updated}, unchanged={unchanged}"
)

# =========================================================
# 8. CONTROLE
# =========================================================

result = pd.read_sql_query("""
SELECT *
FROM Dim_Fiets
ORDER BY fietsnr, source_id, fiets_sk
""", dwh_conn)

print(result)

2026-04-01 12:23:08.506 | INFO     | __main__:<module>:1 - Start ETL Dim_Fiets (SCD Type 2 met business key fietsnr + source_id)
2026-04-01 12:23:08.516 | INFO     | __main__:<module>:42 - Aantal bronrecords: 180
2026-04-01 12:23:08.519 | INFO     | __main__:<module>:65 - Aantal actuele fietsen in DWH: 0
2026-04-01 12:23:08.575 | INFO     | __main__:<module>:105 - Preview | new=180, changed=0, unchanged=0
2026-04-01 12:23:08.621 | SUCCESS  | __main__:<module>:197 - Dim_Fiets klaar | inserted=180, updated=0, unchanged=0


     fiets_sk  fietsnr      source_id              soort         merk  \
0          91        1   Fiets_Inkoop         Stadsfiets   SpeedCycle   
1         166        1  Fiets_Verkoop         Stadsfiets   SpeedCycle   
2         241        1      Onderhoud       Mountainbike   SpeedCycle   
3          92        2   Fiets_Inkoop       Mountainbike   SpeedCycle   
4         167        2  Fiets_Verkoop       Mountainbike   SpeedCycle   
..        ...      ...            ...                ...          ...   
175       238       73  Fiets_Verkoop         Stadsfiets  OranjeFiets   
176       164       74   Fiets_Inkoop          Racefiets  OranjeFiets   
177       239       74  Fiets_Verkoop          Racefiets  OranjeFiets   
178       165       75   Fiets_Inkoop  Elektrische fiets  OranjeFiets   
179       240       75  Fiets_Verkoop  Elektrische fiets  OranjeFiets   

        type  kleur  fabrikantnr  fabrikantnaam  fabrikantadres  \
0    STA-240  Groen            1  SpeedCycle BV  Spoorst

Dim_Monteur (SCD Type 1) MEES

In [5]:
logger.info("Ophalen Monteur uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Accessoire_Verkoop_Monteur m
JOIN Accessoire_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL
                                   
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Fiets_Verkoop_Monteur m
JOIN Fiets_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL

SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Onderhoud_Monteur m
JOIN Onderhoud_Filiaal f ON m.filiaal = f.filiaalnr
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    monteurnr, naam, woonplaats, uurloon,
    filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
    monteur_sk
FROM Dim_Monteur
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

processed_keys = set()

for row in source_data:
    (monteurnr, naam, woonplaats, uurloon,
     filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) = row

    if monteurnr in processed_keys:
        continue
    processed_keys.add(monteurnr)

    now = datetime.now()

    if monteurnr not in dwh_data:
        # nieuwe regels worden toegevoegd
        logger.info(f"Nieuwe monteur: {monteurnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Monteur (
            monteurnr, naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (monteurnr, naam, woonplaats, uurloon,
              filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
              now))

    else:
        # bestaande regels worden overschreven als er wijzigingen zijn
        (_, d_naam, d_woonplaats, d_uurloon,
         d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie,
         monteur_sk) = dwh_data[monteurnr]

        if (naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) != \
           (d_naam, d_woonplaats, d_uurloon,
            d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie):

            logger.info(f"Update monteur (Type 1): {monteurnr}")

            dwh_cursor.execute("""
            UPDATE Dim_Monteur
            SET naam = ?, woonplaats = ?, uurloon = ?,
                filiaalnr = ?, filiaalnaam = ?, filiaaladres = ?, filiaalprovincie = ?
            WHERE monteur_sk = ?
            """, (naam, woonplaats, uurloon,
                  filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
                  monteur_sk))
            
dwh_conn.commit()
logger.info("Monteur dimensie geupdate in DWH.")

2026-04-01 12:16:00.618 | INFO     | __main__:<module>:1 - Ophalen Monteur uit SDM
2026-04-01 12:16:00.626 | INFO     | __main__:<module>:57 - Data opgehaald uit SDM en DWH.
2026-04-01 12:16:00.627 | INFO     | __main__:<module>:73 - Nieuwe monteur: 1
C:\Users\MeesR\AppData\Local\Temp\ipykernel_32276\2513428366.py:75: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_cursor.execute("""
2026-04-01 12:16:00.629 | INFO     | __main__:<module>:73 - Nieuwe monteur: 2
2026-04-01 12:16:00.631 | INFO     | __main__:<module>:73 - Nieuwe monteur: 3
2026-04-01 12:16:00.631 | INFO     | __main__:<module>:73 - Nieuwe monteur: 4
2026-04-01 12:16:00.632 | INFO     | __main__:<module>:73 - Nieuwe monteur: 5
2026-04-01 12:16:00.633 | INFO     | __main__:<module>:73 - Nieuwe monteur: 6
2026-04-01 12:16:00.634 | INFO     | __main__:<module>:73 - Nieuwe monteur: 7
2026-04-01 12:16:00.635 | INFO     | __ma

Fct_Onderhoud MEES

In [6]:
logger.info("Ophalen Onderhoud uit SDM")

sdm_cursor.execute("""
SELECT 
    o.onderhoudnr,
    o.datum,
    o.starttijd,
    o.eindtijd,
    o.fiets,
    o.monteur
FROM Onderhoud o
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("SELECT onderhoudnr FROM Fct_Onderhoud")
bestaande_ids = {row[0] for row in dwh_cursor.fetchall()}

def get_or_create_datum_sk(datum):
    dwh_cursor.execute("SELECT datum_sk FROM Dim_Datum WHERE datum = ?", (datum,))
    if r := dwh_cursor.fetchone():
        return r[0]

    from datetime import datetime
    d = datetime.strptime(datum, "%Y-%m-%d")

    dwh_cursor.execute("""
    INSERT INTO Dim_Datum (datum, year, month, day)
    VALUES (?, ?, ?, ?)
    """, (datum, d.year, d.month, d.day))

    return dwh_cursor.lastrowid


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_monteur_sk(monteurnr):
    dwh_cursor.execute("""
    SELECT monteur_sk FROM Dim_Monteur
    WHERE monteurnr = ? AND eindtijd IS NULL
    """, (monteurnr,))
    result = dwh_cursor.fetchone()
    return result[0] if result else None


logger.info("Start ETL Fct_Onderhoud")

for row in source_data:
    onderhoudnr, datum, starttijd, eindtijd, fietsnr, monteurnr = row

    # skip bestaande
    if onderhoudnr in bestaande_ids:
        continue

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    monteur_sk = get_monteur_sk(monteurnr)

    # Check of alle keys bestaan
    if not all([datum_sk, fiets_sk, monteur_sk]):
        logger.warning(f"SK ontbreekt voor onderhoud {onderhoudnr}")
        continue

    # Insert fact
    dwh_cursor.execute("""
    INSERT INTO Fct_Onderhoud (
        onderhoudnr,
        datum_sk,
        starttijd,
        eindtijd,
        fiets_sk,
        monteur_sk
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (onderhoudnr, datum_sk, starttijd, eindtijd, fiets_sk, monteur_sk))

dwh_conn.commit()
logger.success("Fct_Onderhoud load klaar")

2026-04-01 12:16:10.796 | INFO     | __main__:<module>:1 - Ophalen Onderhoud uit SDM
2026-04-01 12:16:10.799 | INFO     | __main__:<module>:55 - Start ETL Fct_Onderhoud
2026-04-01 12:16:10.801 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 1
2026-04-01 12:16:10.803 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 2
2026-04-01 12:16:10.806 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 3
2026-04-01 12:16:10.808 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 4
2026-04-01 12:16:10.810 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 5
2026-04-01 12:16:10.811 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 6
2026-04-01 12:16:10.812 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 7
2026-04-01 12:16:10.815 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 8
2026-04-01 12:16:10.817 | WARNING  | __main__:<module>:70 - SK ontbreekt voor onderhoud 9
2026-04-01 12:16:10.8

Fct_Inkoop MEES

In [7]:
logger.info("Ophalen Inkoop uit SDM") 

sdm_cursor.execute("""
SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    f.inkoopprijs,
    i.fiets,
    NULL as accessoire
FROM Fiets_Inkoop i
JOIN Fiets_Inkoop_Fiets f ON i.fiets = f.fietsnr

UNION ALL

SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    a.inkoopprijs,
    NULL as fiets,
    i.accessoire
FROM Accessoire_Inkoop i
JOIN Accessoire_Inkoop_Accessoire a ON i.accessoire = a.accessoirenr
""")

source_data = sdm_cursor.fetchall()

# LET OP: we gebruiken nu de SOURCE inkoopnr alleen voor incremental check
dwh_cursor.execute("SELECT inkoop_type, datum_sk, aantal FROM Fct_Inkoop")
bestaande_records = set(dwh_cursor.fetchall())


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_accessoire_sk(accessoirenr):
    if accessoirenr is None:
        return None
    dwh_cursor.execute("""
    SELECT accessoire_sk FROM Dim_Accessoire
    WHERE accessoirenr = ? AND eindtijd IS NULL
    """, (accessoirenr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


logger.info("Start ETL Fct_Inkoop")

for row in source_data:
    source_inkoopnr, datum, aantal, inkoopprijs, fietsnr, accessoirenr = row

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    accessoire_sk = get_accessoire_sk(accessoirenr)

    # bepaal type
    if fiets_sk is not None:
        inkoop_type = "fiets"
    elif accessoire_sk is not None:
        inkoop_type = "accessoire"
    else:
        logger.warning(f"Ongeldige inkoop (geen match): {source_inkoopnr}")
        continue

    # validatie (extra check)
    if fiets_sk is not None and accessoire_sk is not None:
        logger.warning(f"Dubbele koppeling bij {source_inkoopnr}")
        continue

    totaalprijs = aantal * inkoopprijs

    # incremental check (zonder auto-id)
    record_key = (inkoop_type, datum_sk, aantal)
    if record_key in bestaande_records:
        continue

    dwh_cursor.execute("""
    INSERT INTO Fct_Inkoop (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    ))

dwh_conn.commit()
logger.success("Fct_Inkoop load klaar")

2026-04-01 12:16:23.276 | INFO     | __main__:<module>:1 - Ophalen Inkoop uit SDM
2026-04-01 12:16:23.286 | INFO     | __main__:<module>:56 - Start ETL Fct_Inkoop
2026-04-01 12:16:23.288 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 1
2026-04-01 12:16:23.290 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 4
2026-04-01 12:16:23.291 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 9
2026-04-01 12:16:23.291 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 10
2026-04-01 12:16:23.292 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 11
2026-04-01 12:16:23.293 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 12
2026-04-01 12:16:23.295 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 13
2026-04-01 12:16:23.296 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match): 21
2026-04-01 12:16:23.297 | WARNING  | __main__:<module>:71 - Ongeldige inkoop (geen match